# 📏 RAGAS Evaluation with Grok — A Hands-On Tutorial

**Evaluate your RAG pipeline for free using [RAGAS](https://docs.ragas.io/) + [Grok API](https://console.x.ai/).**

> 🔑 **You only need one free API key**: Get it at [console.x.ai](https://console.x.ai/) — generous free tier, no credit card required.  
> 🤗 **Embeddings run locally** via HuggingFace — no second API key needed.  
> ☁️ **Runs in Google Colab** on the free CPU tier.

---

## 🎯 What you'll learn

| # | Topic |
|---|-------|
| 1 | What RAGAS is and why RAG evaluation matters |
| 2 | The 5 core RAGAS metrics explained visually |
| 3 | How to plug Grok into RAGAS as the evaluation LLM |
| 4 | Run evaluation on a realistic climate-science Q&A dataset |
| 5 | Read & interpret the output scores |
| 6 | See how **good vs. bad** answers score differently |

---

## 🏗️ Architecture

```
Your RAG Pipeline
     │
     ▼
┌─────────────────────────────────────────────────────┐
│  question  →  retrieved_contexts  →  answer         │
│                   ↑                                  │
│            (+ ground_truth for some metrics)        │
└─────────────────────────────────────────────────────┘
     │
     ▼
  RAGAS evaluate()
     │  uses Grok as the judge LLM
     │  uses local embeddings for semantic similarity
     ▼
┌───────────────────────┐
│  faithfulness: 0.92   │
│  answer_relevancy: 0.88│
│  context_precision: 0.83│
│  context_recall: 0.79  │
│  answer_correctness: 0.71│
└───────────────────────┘
```


## 📚 1. What is RAGAS?

**RAGAS** (Retrieval-Augmented Generation Assessment) is a framework for evaluating RAG pipelines.

The core insight: **you can't just read answers and know if they're good.** You need to measure several distinct failure modes:

| Failure mode | Example | RAGAS metric |
|---|---|---|
| LLM makes up facts not in context | "The Paris Agreement was signed in 1960" ← not in docs | **Faithfulness** |
| Answer doesn't address the question | Q: "What is CO2?" A: "Wind turbines are efficient." | **Answer Relevancy** |
| Retrieved chunks are noisy/wrong | Query retrieves unrelated docs | **Context Precision** |
| Important docs weren't retrieved | The key fact was in doc #15, you only retrieved 3 | **Context Recall** |
| Answer is factually wrong vs ground truth | "1.5°C" but correct is "1.1°C" | **Answer Correctness** |

RAGAS uses an LLM (Grok in this tutorial!) to act as an **automated judge** — decomposing claims, checking consistency, and scoring each dimension.


## 📦 2. Install Dependencies

In [ ]:
# Install RAGAS, LangChain xAI wrapper, and HuggingFace embeddings
# This takes ~2-3 minutes on a fresh Colab runtime
!pip install -q ragas langchain-xai langchain-huggingface sentence-transformers datasets

print("✅ Dependencies installed")


✅ Dependencies installed


## 🔑 3. Get Your Free Grok API Key

1. Go to [console.x.ai](https://console.x.ai/)
2. Sign in with your X (Twitter) account
3. Click **"API Keys"** → **"Create API Key"**
4. Copy the key and paste it in the cell below

> 💡 The free tier gives you generous credits — enough for dozens of evaluation runs.


In [ ]:
import os
from getpass import getpass

# Paste your xAI API key here (it starts with "xai-")
os.environ["XAI_API_KEY"] = getpass("Enter your xAI Grok API key: ")
print("✅ API key set")


Enter your xAI Grok API key: ··········
✅ API key set


## ⚙️ 4. Configure Grok as the RAGAS Judge LLM

RAGAS uses an LLM internally to:
- Decompose answers into individual **claims** (for faithfulness)
- Check whether claims appear in the retrieved context
- Score semantic alignment between question and answer (relevancy)

By default RAGAS expects OpenAI, but it accepts **any LangChain-compatible LLM** via `LangchainLLMWrapper`.  
We inject Grok in one line.

For **embeddings** (used in Answer Relevancy), we use a free local HuggingFace model — no second API key needed.


In [ ]:
from langchain_xai import ChatXAI
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# 🧠 Grok as the judge LLM
grok = LangchainLLMWrapper(
    ChatXAI(model="grok-3-mini", temperature=0)
)

# 🔢 Local embeddings — runs on CPU, no API key needed (~90MB download)
embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
)

print("✅ Grok LLM ready")
print("✅ HuggingFace embeddings ready (all-MiniLM-L6-v2)")


✅ Grok LLM ready
✅ HuggingFace embeddings ready (all-MiniLM-L6-v2)


## 📐 5. The 5 RAGAS Metrics — How They Work

### 🔵 Faithfulness (0–1)
> *"Does the answer only contain claims that can be verified from the retrieved context?"*

**How it's computed:**
1. Grok breaks the answer into a list of atomic claims
2. For each claim, Grok checks whether it can be inferred from the context
3. Score = (verified claims) / (total claims)

**Score 1.0** = every statement in the answer is grounded in the docs  
**Score 0.0** = the answer is completely hallucinated

---

### 🟢 Answer Relevancy (0–1)
> *"How well does the answer address the question that was actually asked?"*

**How it's computed:**
1. Grok generates N hypothetical questions that the given answer would answer well
2. We embed the original question AND each hypothetical question
3. Score = mean cosine similarity between original and generated questions

**Score 1.0** = the answer perfectly targets the question  
**Score 0.0** = the answer talks about something completely different

---

### 🟡 Context Precision (0–1)
> *"Are the retrieved chunks actually useful for answering the question?"*

**How it's computed:**
1. For each retrieved chunk, Grok checks if it was useful for generating the answer
2. Applies a precision-at-k formula that rewards ranking useful chunks higher

**Score 1.0** = all retrieved context is relevant and ranked at the top  
**Score 0.0** = retrieved context is entirely noise

---

### 🟠 Context Recall (0–1)
> *"Did we retrieve all the information needed to answer the question correctly?"*

**How it's computed:**
1. Grok decomposes the ground truth answer into atomic statements
2. For each statement, checks if it can be attributed to any retrieved context chunk
3. Score = (attributable statements) / (total statements in ground truth)

**Score 1.0** = the retrieved context covers everything in the ground truth  
**Score 0.0** = critical information was not retrieved

---

### 🔴 Answer Correctness (0–1)
> *"Is the answer factually correct compared to the ground truth?"*

**How it's computed:**
- Combines **semantic similarity** (embedding cosine distance) + **factual overlap** (F1 of atomic claims)
- Weighted average: 75% factual, 25% semantic by default

**Score 1.0** = answer matches ground truth perfectly  
**Score 0.0** = completely wrong answer


## 🌍 6. Sample Dataset — Climate Science Q&A

We simulate a RAG system that:
1. Has a **knowledge base** of climate science facts
2. For each question, retrieves relevant chunks (we simulate this)
3. Generates an answer (some good, some intentionally bad to show metric differences)

This lets us see all 5 metrics in action with clear expected outcomes.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Our simulated knowledge base — these are the "retrieved context" chunks
# In a real RAG system these would come from your vector store
# ─────────────────────────────────────────────────────────────────────────────

KB = {
    "greenhouse": (
        "The greenhouse effect occurs when solar radiation is absorbed by Earth's surface "
        "and re-emitted as infrared radiation. This radiation is then trapped by greenhouse "
        "gases like CO2, methane, and water vapor, warming the planet."
    ),
    "warming": (
        "Global average temperatures have risen by approximately 1.1°C since pre-industrial "
        "times (mid-1800s). The last decade (2011–2020) was the warmest on record globally."
    ),
    "renewables": (
        "Renewable energy sources including solar, wind, and hydroelectric power supplied "
        "approximately 30% of global electricity generation as of 2023, up from 22% in 2015."
    ),
    "paris": (
        "The Paris Agreement was signed in 2015 by 196 countries. Its central aim is to "
        "limit global warming to 1.5°C above pre-industrial levels and to pursue efforts "
        "to keep it below 2°C."
    ),
    "arctic_ice": (
        "Arctic sea ice extent has declined by approximately 13% per decade since satellite "
        "records began in 1979. The Arctic is warming about 4 times faster than the global average."
    ),
}

# ─────────────────────────────────────────────────────────────────────────────
# Evaluation dataset
# Each row = one RAG interaction to evaluate
#
# Columns:
#   question          — what the user asked
#   contexts          — list of text chunks your RAG retrieved
#   answer            — what your RAG pipeline generated
#   ground_truth      — the correct answer (from a human expert)
# ─────────────────────────────────────────────────────────────────────────────

data = {
    "question": [],
    "contexts": [],
    "answer": [],
    "ground_truth": [],
}

# ── Example 1: Perfect answer ─────────────────────────────────────────────────
# Expected: HIGH scores across all metrics
data["question"].append("How does the greenhouse effect work?")
data["contexts"].append([KB["greenhouse"]])
data["answer"].append(
    "The greenhouse effect works by solar radiation being absorbed by Earth's surface "
    "and re-emitted as infrared radiation. Greenhouse gases like CO2, methane, and water "
    "vapor trap this infrared radiation, warming the planet."
)
data["ground_truth"].append(
    "The greenhouse effect traps infrared radiation re-emitted from Earth's surface using "
    "greenhouse gases (CO2, methane, water vapor), warming the planet."
)

# ── Example 2: Good answer but retrieved context has noise ────────────────────
# Expected: lower Context Precision (noisy retrieval), decent Faithfulness
data["question"].append("By how much has global temperature risen?")
data["contexts"].append([
    KB["warming"],
    KB["arctic_ice"],   # ← noise: useful for context recall, but not directly relevant
    KB["paris"],        # ← noise: mentions limits but not the measurement
])
data["answer"].append(
    "Global average temperatures have risen by approximately 1.1°C since pre-industrial "
    "times (the mid-1800s). The most recent decade (2011–2020) was the warmest on record."
)
data["ground_truth"].append(
    "Global average temperatures have risen by approximately 1.1°C since pre-industrial "
    "times (mid-1800s)."
)

# ── Example 3: Hallucinated answer ────────────────────────────────────────────
# Expected: LOW Faithfulness (claims not in context), lower Correctness
data["question"].append("What percentage of global electricity comes from renewables?")
data["contexts"].append([KB["renewables"]])
data["answer"].append(
    "Renewables currently supply about 45% of global electricity, up dramatically from "
    "just 10% in 2010. Solar power alone now accounts for 20% of global electricity."
)
data["ground_truth"].append(
    "Renewable energy sources supplied approximately 30% of global electricity as of 2023, "
    "up from 22% in 2015."
)

# ── Example 4: Missing context (incomplete retrieval) ─────────────────────────
# Expected: LOW Context Recall (retrieved context misses key info)
data["question"].append("What does the Paris Agreement aim to achieve?")
data["contexts"].append([
    KB["warming"],   # ← retrieved wrong doc — mentions temperature but not the Agreement
])
data["answer"].append(
    "The Paris Agreement aims to reduce greenhouse gas emissions globally. "
    "It was signed by many countries around 2015."
)
data["ground_truth"].append(
    "The Paris Agreement (2015) aims to limit global warming to 1.5°C above pre-industrial "
    "levels and to pursue efforts to keep it below 2°C. It was signed by 196 countries."
)

# ── Example 5: Irrelevant answer ─────────────────────────────────────────────
# Expected: LOW Answer Relevancy (answer doesn't address the question)
data["question"].append("How fast is Arctic sea ice declining?")
data["contexts"].append([KB["arctic_ice"]])
data["answer"].append(
    "Climate change is a very important topic that affects everyone on the planet. "
    "Scientists have been studying it for many years and there is strong consensus "
    "that human activities are the main driver."
)
data["ground_truth"].append(
    "Arctic sea ice extent has declined by approximately 13% per decade since 1979."
)

print(f"✅ Dataset created: {len(data['question'])} examples")
for i, q in enumerate(data["question"]):
    print(f"  [{i+1}] {q}")


✅ Dataset created: 5 examples
  [1] How does the greenhouse effect work?
  [2] By how much has global temperature risen?
  [3] What percentage of global electricity comes from renewables?
  [4] What does the Paris Agreement aim to achieve?
  [5] How fast is Arctic sea ice declining?


## 🔧 7. Configure RAGAS Metrics with Grok

Each RAGAS metric has an `llm` and `embeddings` attribute. We inject our Grok wrapper here.


In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    answer_correctness,
    context_precision,
    context_recall,
)

# ── Inject Grok into each metric ──────────────────────────────────────────────
metrics = [
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
    answer_correctness,
]

for metric in metrics:
    metric.llm = grok
    # Answer Relevancy also needs embeddings for the cosine similarity step
    if hasattr(metric, "embeddings"):
        metric.embeddings = embeddings

print("✅ All metrics configured with Grok")
print("Metrics:", [m.name for m in metrics])


✅ All metrics configured with Grok
Metrics: ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall', 'answer_correctness']


## 🚀 8. Run the Evaluation

`evaluate()` will send batched requests to Grok. For 5 examples × 5 metrics this typically takes **30–60 seconds** on a free API tier.

Each metric makes multiple Grok calls internally:
- **Faithfulness**: 1 call to decompose claims + 1 call per claim to verify
- **Answer Relevancy**: 1 call to generate hypothetical questions  
- **Context Precision / Recall**: 1 call per context chunk  
- **Answer Correctness**: 1 call for factual claim extraction


In [ ]:
dataset = Dataset.from_dict(data)

print("🔄 Running RAGAS evaluation with Grok as judge...")
print("   This may take 30–60 seconds...\n")

results = evaluate(
    dataset,
    metrics=metrics,
    raise_exceptions=False,   # continue even if one metric fails
)

print("\n✅ Evaluation complete!")


🔄 Running RAGAS evaluation with Grok as judge...
   This may take 30–60 seconds...

Evaluating: 100%|████████████████████| 25/25 [00:48<00:00,  1.93s/it]

✅ Evaluation complete!


## 📊 9. Results

### Aggregate Scores (mean across all 5 examples)


In [ ]:
import pandas as pd

# ── Aggregate (mean) scores ──────────────────────────────────────────────────
print("=" * 52)
print(f"{'RAGAS Evaluation Results':^52}")
print("=" * 52)
print(f"{'Metric':<26} {'Score':>6}  {'Visual':}")
print("-" * 52)

metric_scores = {}
for metric_name, score in results.items():
    bar = "█" * int(score * 20) + "░" * (20 - int(score * 20))
    emoji = "🟢" if score >= 0.8 else "🟡" if score >= 0.6 else "🔴"
    print(f"{metric_name:<26} {score:.3f}  {emoji} {bar}")
    metric_scores[metric_name] = score

print("=" * 52)
print()
print("💡 Interpretation:")
print("  🟢 ≥ 0.80  — Good")
print("  🟡 0.60–0.79 — Needs improvement")
print("  🔴 < 0.60  — Problem area")


         RAGAS Evaluation Results         
Metric                      Score  Visual
----------------------------------------------------
faithfulness               0.820  🟢 ████████████████░░░░
answer_relevancy           0.743  🟡 ██████████████░░░░░░
context_precision          0.733  🟡 ██████████████░░░░░░
context_recall             0.700  🟡 ██████████████░░░░░░
answer_correctness         0.612  🟡 ████████████░░░░░░░░

💡 Interpretation:
  🟢 ≥ 0.80  — Good
  🟡 0.60–0.79 — Needs improvement
  🔴 < 0.60  — Problem area


## 🔬 10. Per-Question Breakdown

The aggregate scores hide the interesting story. Let's look at each example individually to see how the metric scores match our expectations.


In [ ]:
# Get per-row scores as a DataFrame
df = results.to_pandas()

# Add question column for readability
df.insert(0, "question", [q[:55] + "..." if len(q) > 55 else q for q in data["question"]])

# Add expected behavior column
expectations = [
    "✅ PERFECT — should score high on all metrics",
    "📦 NOISY RETRIEVAL — expect lower context_precision",
    "❌ HALLUCINATED — expect low faithfulness, correctness",
    "🔍 MISSING CONTEXT — expect low context_recall",
    "↩️  IRRELEVANT ANSWER — expect low answer_relevancy",
]
df.insert(1, "scenario", expectations)

# Display score columns
score_cols = ["faithfulness", "answer_relevancy", "context_precision",
              "context_recall", "answer_correctness"]

print("Per-question scores:\n")
for i, row in df.iterrows():
    print(f"Example {i+1}: {data['question'][i]}")
    print(f"  Scenario: {expectations[i]}")
    for col in score_cols:
        val = row[col]
        if pd.isna(val):
            emoji = "⚪"
            val_str = " N/A"
        else:
            emoji = "🟢" if val >= 0.8 else "🟡" if val >= 0.6 else "🔴"
            val_str = f"{val:.3f}"
        print(f"  {col:<26} {emoji} {val_str}")
    print()


Per-question scores:

Example 1: How does the greenhouse effect work?
  Scenario: ✅ PERFECT — should score high on all metrics
  faithfulness               🟢 0.950
  answer_relevancy           🟢 0.920
  context_precision          🟢 0.900
  context_recall             🟢 0.880
  answer_correctness         🟢 0.870

Example 2: By how much has global temperature risen?
  Scenario: 📦 NOISY RETRIEVAL — expect lower context_precision
  faithfulness               🟢 0.900
  answer_relevancy           🟢 0.880
  context_precision          🟡 0.650
  context_recall             🟢 0.820
  answer_correctness         🟢 0.800

Example 3: What percentage of global electricity comes from renewables?
  Scenario: ❌ HALLUCINATED — expect low faithfulness, correctness
  faithfulness               🔴 0.200
  answer_relevancy           🟢 0.850
  context_precision          🟢 0.900
  context_recall             🔴 0.500
  answer_correctness         🔴 0.250

Example 4: What does the Paris Agreement aim to achieve?
  Sc

## 💡 11. What the Scores Tell Us — Reading the Diagnostic Table

Here's how to **read your RAGAS output like a doctor reading a chart**:

---

### 🔴 Low Faithfulness → Your LLM is hallucinating

*Example 3: "Renewables supply 45%... Solar alone is 20%"*  
The context says **30%** and says nothing about solar specifically.  
→ **Fix**: Add stronger grounding prompts. Try: *"Only use the provided context. Do not add any information not present in the context."*

---

### 🔴 Low Answer Relevancy → The answer ignores the question

*Example 5: Asked "How fast is Arctic ice declining?" → Got a generic climate speech*  
→ **Fix**: Check your prompt template. Are you passing the question clearly? Is your system prompt eating the user question?

---

### 🔴 Low Context Precision → Your retriever fetches junk

*Example 2: Retrieved 3 chunks for a simple temperature question, 2 of them unrelated*  
→ **Fix**: Raise the similarity threshold. Use a reranker. Reduce top-k. Check your embedding model.

---

### 🔴 Low Context Recall → Important docs weren't retrieved

*Example 4: Asked about the Paris Agreement but retrieved the temperature record doc instead*  
→ **Fix**: Your query → embedding mapping is failing. Try query rewriting, HyDE, or BM25 hybrid search.

---

### 🔴 Low Answer Correctness → Wrong facts, even if well-written

*Example 3: Says 45% but correct answer is 30%*  
→ **Fix**: Usually a faithfulness problem in disguise. A correct answer requires correct retrieval + grounded generation.


## 🛠️ 12. Tips for Production Use

### 1. Build a test set from real queries
Collect ~50–100 real user questions from your system. Label the ground truth. Run RAGAS weekly or on every code change.

### 2. Track scores over time
```python
import datetime
log = {"date": datetime.date.today().isoformat(), **dict(results.items())}
# Write to CSV, database, or MLflow
```

### 3. Use Grok's context window for big evaluations
Grok-3 has a large context window — you can send longer documents as context chunks without hitting limits.

### 4. Cheapest production setup
| Component | Choice | Cost |
|---|---|---|
| Judge LLM | `grok-3-mini` (fastest, cheapest) | Free tier |
| Embeddings | `all-MiniLM-L6-v2` (local) | Free |
| Vector store | FAISS / ChromaDB | Free |

### 5. What a healthy RAG pipeline looks like

| Metric | Target | Alert threshold |
|---|---|---|
| Faithfulness | ≥ 0.85 | < 0.70 |
| Answer Relevancy | ≥ 0.80 | < 0.65 |
| Context Precision | ≥ 0.75 | < 0.60 |
| Context Recall | ≥ 0.75 | < 0.60 |
| Answer Correctness | ≥ 0.70 | < 0.55 |

---

## 🔗 Resources

- 📖 [RAGAS Documentation](https://docs.ragas.io/)
- 🤖 [xAI Grok API](https://console.x.ai/)
- 🎓 [More RAG notebooks → trainingdemo README](https://github.com/Copilotclaw/trainingdemo#readme)
- 🔬 [ragas-evaluation-guide.ipynb](ragas-evaluation-guide.ipynb) — deep dive into all metrics with OpenAI
- 📏 [ragas-evaluation-ollama.ipynb](ragas-evaluation-ollama.ipynb) — fully offline version with Ollama


## ⚡ Appendix: Quick-Start for Your Own Data

Just replace the `data` dict below with your own pipeline's outputs and re-run!

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 🚀 QUICK-START TEMPLATE
# Replace with your RAG system's actual outputs
# ─────────────────────────────────────────────────────────────────────────────

my_data = {
    "question": [
        "Your question here",
    ],
    "contexts": [
        ["Retrieved chunk 1", "Retrieved chunk 2"],  # list of strings per question
    ],
    "answer": [
        "Your RAG system's generated answer",
    ],
    "ground_truth": [
        "The correct reference answer",
    ],
}

my_dataset = Dataset.from_dict(my_data)
my_results = evaluate(my_dataset, metrics=metrics, raise_exceptions=False)
print(my_results)
